# Python Runtime Pitfalls, Code Smells, and Clean Refactoring

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

Writing reliable software for scientific computing and artificial intelligence requires moving beyond code that merely executes to code that is robust, defensive, maintainable, and aligned with Pythonic standards.

### Objectives
1. Identify and remediate critical runtime bugs: unbounded recursion, mutation during iteration, string formatting oversights, and unhandled key lookups.
2. Eliminate common PEP 8 style violations and code smells: variable shadowing, magic numbers, dead code, global state, and resource leaks.
3. Understand the mutable default argument trap and operator precedence pitfalls.
4. Master modern Pythonic idioms: tuple unpacking, chained comparisons, context managers, and dictionary dispatching.


## 1. Critical Runtime Pitfalls

In this section, we analyze four major failure modes encountered in computational scripts:
- **Unbounded recursion (`RecursionError`):** Missing base conditions causing call stack exhaustion.
- **Mutating sequences during iteration:** Appending elements causing infinite loops or removing elements causing skipped items due to index shifting.
- **String formatting omissions:** Omitting the `f` prefix in formatted string literals.
- **Key lookup failures (`KeyError`):** Directly accessing uninitialized keys during in-place dictionary updates.


In [ ]:
import sys


# 1. Unbounded Recursion vs Base Condition
def recursive_infinite(n: int) -> int:
    # Buggy: missing base condition leads to RecursionError
    return recursive_infinite(n)


def recursive_factorial(n: int) -> int:
    # Defensive fix: validated base condition
    if n < 0:
        raise ValueError("Factorial is not defined for negative integers.")
    if n in (0, 1):
        return 1
    return n * recursive_factorial(n - 1)


print("Factorial of 5:", recursive_factorial(5))
print("Current recursion limit:", sys.getrecursionlimit())
assert recursive_factorial(5) == 120

In [ ]:
# 2. Mutating a List While Iterating Over It

# Bug demonstration: Appending inside iteration causes an infinite loop:
# numbers = [1, 2, 3, 4, 5]
# for num in numbers:
#     if num == 3:
#         numbers.append(3)  # Infinite loop!

# Bug demonstration: Removing inside iteration causes skipped elements:
fruits = ["apple", "banana", "banana", "peach"]
# for fruit in fruits:
#     if fruit == "banana":
#         fruits.remove(fruit) # Skips subsequent elements due to internal index shift!

# Idiom 1: List comprehension (pure functional, non-mutating)
filtered_fruits = [f for f in fruits if f != "banana"]
print("Filtered via comprehension:", filtered_fruits)
assert filtered_fruits == ["apple", "peach"]

# Idiom 2: Iterating over a shallow copy
fruits_copy = ["apple", "banana", "banana", "peach"]
for f in fruits_copy[:]:
    if f == "banana":
        fruits_copy.remove(f)
print("Filtered via slice copy:", fruits_copy)
assert fruits_copy == ["apple", "peach"]

In [ ]:
# 3. String Formatting Pitfall: Missing 'f' Prefix
name = "Ada"
age = 36
buggy_str = "Hello {name}. You are {age} years old."
fixed_str = f"Hello {name}. You are {age} years old."
print("Buggy string literal:", buggy_str)
print("Fixed f-string literal:", fixed_str)
assert fixed_str == "Hello Ada. You are 36 years old."

# 4. Dictionary KeyError vs Safe Lookups
inventory = {"apple": 5}
# Buggy: inventory["banana"] += 2  # Raises KeyError!

# Fix A: .get() with fallback
inventory["banana"] = inventory.get("banana", 0) + 2

# Fix B: .setdefault()
inventory.setdefault("peach", 0)
inventory["peach"] += 10

print("Updated inventory:", inventory)
assert inventory["banana"] == 2
assert inventory["peach"] == 10

## 2. PEP 8 Antipatterns and Code Smells

Code formatting directly impacts code maintainability and debugging speed. Here we review and correct recurring code smells:
- Inconsistent whitespace and operator padding.
- Shadowing built-in names (e.g., `len = 3`).
- Magic numbers without named constants.
- Dead code after return statements.
- Global variable mutation vs functional return values.
- Mutable default arguments in function signatures.


In [ ]:
import math

# Smell 1: Magic numbers vs Named Constants
# Bad: area = radius * radius * 3.14159
PI = math.pi


def compute_circle_area(radius: float) -> float:
    if radius < 0:
        raise ValueError("Radius must be non-negative.")
    return PI * (radius**2)


print("Area for radius 4.0:", compute_circle_area(4.0))
assert math.isclose(compute_circle_area(4.0), 16.0 * math.pi)

# Smell 2: Shadowing built-in functions
# Bad: len = 3  (Shadows built-in len())
item_count = 3
sample_items = ["data1", "data2", "data3"]
print("Built-in len still intact:", len(sample_items))
assert len(sample_items) == item_count

In [ ]:
# Smell 3: Dead Code & Single Exit
def relu_clean(x: float) -> float:
    """Compute the Rectified Linear Unit activation."""
    return max(0.0, float(x))


assert relu_clean(-5.0) == 0.0
assert relu_clean(3.5) == 3.5


# Smell 4: Avoiding Mutable Default Arguments
# Bad: def add_item(item, basket=[]) -> mutable list shared across all calls!
def add_item_safe(item: str, basket: list[str] | None = None) -> list[str]:
    """Safely append item using None as default sentinel."""
    if basket is None:
        basket = []
    basket.append(item)
    return basket


call_1 = add_item_safe("apple")
call_2 = add_item_safe("banana")
print("Call 1 basket:", call_1)
print("Call 2 basket:", call_2)
assert call_1 == ["apple"]
assert call_2 == ["banana"]

In [ ]:
# Smell 5: Operator Precedence in Algebraic Equations
# Bug in quadratic formula: (-b + math.sqrt(delta)) / 2 * a
# In Python, / and * have identical precedence, evaluating left to right:
# ((-b + sqrt(delta)) / 2) * a != (-b + sqrt(delta)) / (2 * a)


def solve_quadratic(a: float, b: float, c: float) -> tuple[float, ...]:
    """Solve ax^2 + bx + c = 0 with correct mathematical precedence."""
    if a == 0:
        if b == 0:
            return ()
        return (-c / b,)

    delta = b**2 - 4 * a * c
    if delta < 0:
        return ()
    if delta == 0:
        return (-b / (2 * a),)

    sqrt_delta = math.sqrt(delta)
    x1 = (-b - sqrt_delta) / (2 * a)
    x2 = (-b + sqrt_delta) / (2 * a)
    return tuple(sorted((x1, x2)))


roots = solve_quadratic(1.0, -3.0, 2.0)
print("Roots of x^2 - 3x + 2 = 0:", roots)
assert roots == (1.0, 2.0)

## 3. Idiomatic Python Patterns

Clean Python leverages expressive language features that reduce boilerplate, eliminate state mutations, and clarify intent:
- Tuple unpacking and in-place variable swaps.
- Chained comparisons (`1 < x < 10`).
- Dictionary dispatching in place of lengthy `if-elif` cascades.
- `enumerate()` and `zip()` for coordinated iteration.


In [ ]:
from collections import Counter
from collections.abc import Callable

# 1. In-place variable swap
x, y = 10, 20
x, y = y, x
assert (x, y) == (20, 10)

# 2. String reversal via slicing
word = "algorithm"
reversed_word = word[::-1]
assert reversed_word == "mhtirogla"

# 3. Chained comparison
value = 7
is_in_bounds = 1 < value < 10
assert is_in_bounds is True

# 4. Dictionary Dispatch Table
BinaryOp = Callable[[float, float], float]
OPERATIONS: dict[str, BinaryOp] = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
    "div": lambda a, b: a / b if b != 0 else float("nan"),
}

print("Dispatch addition:", OPERATIONS["add"](10, 5))
print("Dispatch multiplication:", OPERATIONS["mul"](10, 5))
assert OPERATIONS["add"](10, 5) == 15
assert OPERATIONS["mul"](10, 5) == 50

# 5. Frequency counting using Counter
tokens = ["python", "data", "python", "model", "data", "python"]
counts = Counter(tokens)
most_common_word, freq = counts.most_common(1)[0]
print(f"Most frequent: {most_common_word} ({freq} times)")
assert most_common_word == "python" and freq == 3